# Quantum Gate and Algorithm Reference Library

**Project:** Quantum Algorithm Testing Framework

## Purpose

This notebook documents commonly used quantum gates and reference quantum algorithms with known expected outputs.

The reference circuits and results will help validate our quantum algorithm testing framework by comparing actual results against expected behavior.

## Contents

1. Basic Quantum Gates
2. Rotation Gates
3. Deutsch–Jozsa Algorithm
4. Random Bit and Random Byte
5. Bernstein–Vazirani Algorithm
6. Grover's Search Algorithm
7. Quantum Fourier Transform (QFT and inverse QFT)
8. Quantum Phase Estimation (phase-gate reference)
9. Expected Outputs and Verification
10. Book Reference Circuits and Coverage
11. Entanglement, Swap Test and Teleportation
12. Reversible Arithmetic and Conditional Logic
13. Multi-Solution Grover and QFT Signals

Reusable circuits are defined as functions. Loading this notebook defines them without running examples. Call `hadamard_circuit()` or `x_circuit()` to build a circuit.


## 1. Basic Quantum Gates

### Pauli-X Gate

The Pauli-X gate flips a qubit from |0⟩ to |1⟩
or from |1⟩ to |0⟩.

Expected output:
- Input |0⟩ → Output |1⟩
- Input |1⟩ → Output |0⟩

In [ ]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector


def x_circuit():
    """Return a one-qubit X circuit without measurements."""
    qc = QuantumCircuit(1)
    qc.x(0)
    return qc


### Pauli-Y Gate

The Pauli-Y gate performs a bit flip and introduces a phase change.

**Expected behavior:**

- Y|0⟩ = i|1⟩
- Y|1⟩ = −i|0⟩

The Pauli-Y gate is useful for manipulating quantum states and testing phase-sensitive operations.

In [ ]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector


def y_circuit():
    """Return a one-qubit Y circuit without measurements."""
    qc = QuantumCircuit(1)
    qc.y(0)
    return qc


### Pauli-Z Gate

The Pauli-Z gate changes the phase of a qubit without changing its measurement probabilities in the computational basis.

**Expected behavior:**
- Z|0⟩ = |0⟩
- Z|1⟩ = -|1⟩

The Z gate is commonly used for phase manipulation in quantum algorithms.

In [ ]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector


def z_circuit():
    """Return a one-qubit Z circuit without measurements."""
    qc = QuantumCircuit(1)
    qc.z(0)
    return qc


### Hadamard (H) Gate

The Hadamard gate creates an equal superposition of |0⟩ and |1⟩.

**Expected behavior:**

- H|0⟩ = (|0⟩ + |1⟩) / √2
- H|1⟩ = (|0⟩ − |1⟩) / √2

When applied to |0⟩, the probabilities of measuring 0 and 1 are both 50%.

**Common uses:**
- Bell-state preparation
- Deutsch–Jozsa algorithm
- Quantum Fourier Transform (QFT)
- Grover's search algorithm

In [ ]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector


def hadamard_circuit():
    """Return a one-qubit H circuit without measurements."""
    qc = QuantumCircuit(1)
    qc.h(0)
    return qc


## Rotation Gates

Rotation gates rotate a qubit's state around an axis of the Bloch sphere.
Unlike the Pauli gates, rotation gates accept an angle as a parameter.

The three main rotation gates are RX, RY, and RZ.

### RX Gate

The RX gate rotates a qubit around the X-axis by an angle θ.

**Expected behavior:**
- RX(0)|0⟩ = |0⟩
- RX(π)|0⟩ = -i|1⟩
- RX(π/2)|0⟩ = (|0⟩ - i|1⟩)/√2

**Common uses:**
- Quantum state preparation
- Variational quantum circuits
- Testing parameterized quantum gates

In [ ]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
import numpy as np


def rx_circuit(angle=np.pi):
    """Return a one-qubit RX circuit without measurements."""
    qc = QuantumCircuit(1)
    qc.rx(angle, 0)
    return qc


## Deutsch–Jozsa Algorithm

Distinguishes constant from balanced functions. These example oracles are
`constant_0`, `constant_1`, and `balanced` (input parity).

Read only the input qubits, excluding the last, auxiliary qubit:
all zeros means constant; any nonzero result means balanced.
Returns an unmeasured circuit, like the other library functions.

```python
circuit = deutsch_jozsa_circuit()  # Two input qubits, balanced oracle.
circuit = deutsch_jozsa_circuit(3, "constant_1")
```


In [ ]:
def deutsch_jozsa_circuit(num_input_qubits=2, oracle_type="balanced"):
    """Build Deutsch–Jozsa with a constant or parity oracle; no measurements."""
    if type(num_input_qubits) is not int or num_input_qubits < 1:
        raise ValueError("num_input_qubits must be a positive integer.")
    if oracle_type not in ("constant_0", "constant_1", "balanced"):
        raise ValueError("oracle_type must be constant_0, constant_1, or balanced.")

    auxiliary = num_input_qubits
    qc = QuantumCircuit(num_input_qubits + 1)

    # Prepare inputs in |+> and the auxiliary qubit in |->.
    qc.x(auxiliary)
    qc.h(range(num_input_qubits + 1))

    # Oracle: |x, y> -> |x, y XOR f(x)>.
    if oracle_type == "constant_1":
        qc.x(auxiliary)
    elif oracle_type == "balanced":
        for qubit in range(num_input_qubits):
            qc.cx(qubit, auxiliary)
    # constant_0 leaves the auxiliary qubit unchanged.

    qc.h(range(num_input_qubits))
    return qc


## Random Bit

One qubit with equal probabilities of measuring 0 or 1.


In [ ]:
def random_bit_circuit():
    """Return an unmeasured circuit for one uniformly random bit."""
    qc = QuantumCircuit(1)
    qc.h(0)
    return qc


## Random Byte

Eight qubits with equal probabilities for all 256 bitstrings (0–255).


In [ ]:
def random_byte_circuit():
    """Return an unmeasured circuit for eight independent random bits."""
    qc = QuantumCircuit(8)
    qc.h(range(8))
    return qc


## Bernstein–Vazirani Algorithm

Recover a hidden bitstring `s` using one query to the parity oracle
`f(x) = s · x mod 2`. The last qubit is an auxiliary qubit in |−⟩.
Qiskit displays strings as `q[n-1]...q[0]`; read only the input register.
For `secret="101"`, its ideal output is `101` with probability 1.


In [ ]:
## @brief Recover a hidden binary string using a parity oracle and phase kickback.
# @param secret Nonempty binary string in displayed order, highest input qubit first.
# @return QuantumCircuit with len(secret) input qubits and one auxiliary qubit; no measurements.
# @exception ValueError If secret is not a nonempty binary string.
# @note Input-register probabilities are concentrated on secret. Exclude the last auxiliary qubit.
# @par Example
# bernstein_vazirani_circuit("101") yields input-register outcome 101 with probability 1.
def bernstein_vazirani_circuit(secret="101"):
    if not isinstance(secret, str) or not secret or any(bit not in "01" for bit in secret):
        raise ValueError("secret must be a nonempty binary string.")
    size = len(secret)
    qc = QuantumCircuit(size + 1)
    qc.x(size)
    qc.h(range(size + 1))
    for qubit, bit in enumerate(reversed(secret)):
        if bit == "1":
            qc.cx(qubit, size)
    qc.h(range(size))
    return qc


## Grover's Search Algorithm

Amplify one marked computational basis state using a phase oracle followed by
inversion about the mean. This reference constructs an oracle from a known
bitstring for testing; it does not discover an unknown oracle's implementation.
The default `marked_state="11"` needs one iteration and succeeds with probability 1.
For N states and r iterations, success probability is
`sin²((2r+1) asin(1/sqrt(N)))`; it need not be 1 for other sizes.


In [ ]:
## @brief Build single-solution Grover search with a bitstring phase oracle.
# @param marked_state Nonempty binary string, highest qubit first in Qiskit display order.
# @param iterations Nonnegative integer, or None to choose floor(pi/(4*asin(1/sqrt(N)))).
# @return Unmeasured QuantumCircuit on len(marked_state) qubits.
# @exception ValueError If the bitstring or iteration count is invalid.
# @note N = 2**len(marked_state). Zero iterations returns uniform superposition.
# @par Example
# grover_circuit("10") yields 10 with probability 1 in an ideal simulation.
def grover_circuit(marked_state="11", iterations=None):
    if not isinstance(marked_state, str) or not marked_state or any(bit not in "01" for bit in marked_state):
        raise ValueError("marked_state must be a nonempty binary string.")
    size = len(marked_state)
    if iterations is None:
        iterations = int(np.floor(np.pi / (4 * np.arcsin(1 / np.sqrt(2**size)))))
    if type(iterations) is not int or iterations < 0:
        raise ValueError("iterations must be a nonnegative integer or None.")
    qc = QuantumCircuit(size)
    qc.h(range(size))
    for _ in range(iterations):
        # Map the marked state to all ones, then apply a multi-controlled Z.
        zeros = [q for q, bit in enumerate(reversed(marked_state)) if bit == "0"]
        for qubit in zeros:
            qc.x(qubit)
        if size == 1:
            qc.z(0)
        else:
            qc.h(size - 1)
            qc.mcx(list(range(size - 1)), size - 1)
            qc.h(size - 1)
        for qubit in zeros:
            qc.x(qubit)
        # Diffusion, up to an irrelevant global phase.
        qc.h(range(size))
        qc.x(range(size))
        if size == 1:
            qc.z(0)
        else:
            qc.h(size - 1)
            qc.mcx(list(range(size - 1)), size - 1)
            qc.h(size - 1)
        qc.x(range(size))
        qc.h(range(size))
    return qc


## Quantum Fourier Transform

The forward QFT maps |x⟩ to `sum_y exp(2πi*x*y/N)|y⟩/sqrt(N)`.
Final swaps preserve Qiskit's little-endian integer ordering. This is a reusable
transform: prepare an input and compose this circuit onto it. On |0...0⟩ it
produces a uniform superposition. Applying QFT then inverse QFT restores any input.

Reference: [IBM Quantum: phase estimation and QFT](https://quantum.cloud.ibm.com/learning/en/courses/fundamentals-of-quantum-algorithms/phase-estimation-and-factoring/phase-estimation-procedure).


In [ ]:
## @brief Construct the exact QFT or its inverse, including final bit-order swaps.
# @param num_qubits Positive integer specifying the transform register size.
# @param inverse Boolean selecting the inverse transform when True.
# @return Unmeasured QuantumCircuit implementing the transform with no input preparation.
# @exception ValueError If num_qubits is not positive or inverse is not boolean.
# @note The forward transform uses positive Fourier phases and O(num_qubits**2) gates.
# @par Example
# qft_circuit(3).compose(qft_circuit(3, inverse=True)) implements identity.
def qft_circuit(num_qubits=3, inverse=False):
    if type(num_qubits) is not int or num_qubits < 1:
        raise ValueError("num_qubits must be a positive integer.")
    if type(inverse) is not bool:
        raise ValueError("inverse must be a boolean.")
    qc = QuantumCircuit(num_qubits, name="QFT")
    for target in reversed(range(num_qubits)):
        qc.h(target)
        for control in reversed(range(target)):
            qc.cp(np.pi / 2**(target - control), control, target)
    for qubit in range(num_qubits // 2):
        qc.swap(qubit, num_qubits - qubit - 1)
    return qc.inverse() if inverse else qc


## Quantum Phase Estimation

Estimate the eigenphase of `U = P(2π*phase)` using its eigenstate |1⟩.
The first `num_counting_qubits` qubits store the estimate; the last is the target.
Read only the counting register and divide its integer result by `2**num_counting_qubits`.
The default phase 0.125 with three counting qubits gives `001` with probability 1.
Nonrepresentable phases yield a distribution around nearby binary estimates.
This is a known-unitary reference example, not an API for arbitrary unitaries.


In [ ]:
## @brief Estimate a phase-gate eigenphase using controlled powers and inverse QFT.
# @param phase Finite real number in [0, 1), measured in turns rather than radians.
# @param num_counting_qubits Positive integer controlling precision (spacing 2**(-n)).
# @return Unmeasured QuantumCircuit with n counting qubits followed by one target in |1>.
# @exception ValueError If phase or the counting-register size is invalid.
# @note Read only qubits 0 through n-1. Exactly representable phases yield integer phase*2**n.
# @par Example
# phase_estimation_circuit(0.375, 3) yields counting-register outcome 011 with probability 1.
def phase_estimation_circuit(phase=0.125, num_counting_qubits=3):
    from numbers import Real

    if isinstance(phase, (bool, np.bool_)) or not isinstance(phase, Real) or not np.isfinite(phase) or not 0 <= phase < 1:
        raise ValueError("phase must be a finite real number in [0, 1).")
    if type(num_counting_qubits) is not int or num_counting_qubits < 1:
        raise ValueError("num_counting_qubits must be a positive integer.")
    target = num_counting_qubits
    qc = QuantumCircuit(target + 1)
    qc.x(target)
    qc.h(range(target))
    for control in range(target):
        qc.cp(2 * np.pi * phase * 2**control, control, target)
    qc.compose(qft_circuit(target, inverse=True), qubits=list(range(target)), inplace=True)
    return qc


# Programming Quantum Computers: Reference Circuits

The following original implementations cover the main circuit families in the
[book's public Qiskit examples](https://github.com/oreilly-qc/oreilly-qc.github.io/tree/master/samples/Qiskit)
by Eric Johnston, Nic Harrigan and Mercedes Gimeno-Segovia (O'Reilly Media).
They modernize the examples' concepts for this project's Qiskit 2.x environment;
they are not verbatim ports. Existing Grover and QFT builders above remain reusable.

**Contract:** builders return fresh circuits without measurements or backend jobs.
Integer registers use little-endian qubits; displayed strings put the highest qubit
first. Angles are radians. Parameters prepare reference inputs; arithmetic operations
remain reversible and work on superpositions when composed without preparation.
The example-output table below identifies the register to inspect.

| Book examples | Library coverage |
|---|---|
| 2-1, 2-2 random bits | `random_bit_circuit`, `random_byte_circuit` |
| 2-3 root of NOT | `sqrt_x_circuit` |
| 3-2 entanglement, 3-6 correlated remote randomness | `bell_state_circuit` |
| 3-3 phase kickback | `phase_kickback_circuit` (target explicitly prepared in an eigenstate) |
| 3-4 swap test | `swap_test_circuit` (standard P(ancilla=0) convention) |
| 3-5 custom controlled phase, 5-5 conditional phase | `conditional_phase_circuit` |
| 4-1, 4-2 teleportation | `teleportation_circuit` (coherent deferred-measurement version) |
| 5-1 increment/decrement | `increment_circuit` |
| 5-2 integer addition | `quantum_addition_circuit` |
| 5-3 add squared integer | `add_squared_circuit` |
| 5-4 conditional execution | `conditional_increment_circuit` |
| 5-6 reversible CNOT logic | `reversible_xor_circuit` |
| 6-1 iterations, 6-2 amplification oscillation | existing `grover_circuit(iterations=...)` |
| 6-3 multiple marked states | `grover_multi_circuit` |
| 7-1 through 7-8 Fourier demonstrations | existing `qft_circuit`, plus `qft_signal_circuit` for phase/square signals |

GHZ preparation is included as a useful extension of the Bell-state example.
This is coverage of circuit families, not every book visualization or workflow.
Spy Hunter's interception/measurement workflow (2-4), separate separability demo
(3-1), flying-teleportation visualization and the exact Chapter 7 signal variants
are not reproduced individually. No Shor, VQE or QAOA implementation is implied.

In [ ]:
def _validate_register_value(num_qubits, value, label="value"):
    """Validate a small unsigned reference input without coercing booleans."""
    if type(num_qubits) is not int or num_qubits < 1:
        raise ValueError("num_qubits must be a positive integer.")
    if type(value) is not int or not 0 <= value < 2**num_qubits:
        raise ValueError(f"{label} must be an integer in [0, 2**num_qubits).")


def _prepare_integer(qc, qubits, value):
    """Prepare an unsigned basis value on a little-endian register."""
    for bit, qubit in enumerate(qubits):
        if value & (1 << bit):
            qc.x(qubit)


def _append_increment(qc, targets, controls=(), shift=0):
    """Add 2**shift modulo register size, optionally controlled; no scratch qubits."""
    for bit in reversed(range(shift, len(targets))):
        conditions = list(controls) + list(targets[shift:bit])
        if conditions:
            qc.mcx(conditions, targets[bit])
        else:
            qc.x(targets[bit])


def _append_phase_flip(qc, qubits):
    """Change the sign of the all-ones state of a nonempty register."""
    if len(qubits) == 1:
        qc.z(qubits[0])
    else:
        qc.h(qubits[-1])
        qc.mcx(list(qubits[:-1]), qubits[-1])
        qc.h(qubits[-1])


def _validate_angle(angle, label):
    """Require a finite real angle in radians."""
    from numbers import Real
    if isinstance(angle, (bool, np.bool_)) or not isinstance(angle, Real) or not np.isfinite(angle):
        raise ValueError(f"{label} must be a finite real number.")

## Square Root of NOT

A square-root-X gate produces equal computational-basis probabilities on |0⟩.
Two applications equal X exactly. Qiskit's `sx` chooses the positive square root;
the book's negative-angle example chooses its inverse, which also squares to X.

In [ ]:
## @brief Build repeated square-root-X operations on one qubit (book example 2-3).
# @param repetitions Nonnegative integer number of SX gates; default two demonstrates NOT.
# @return Unmeasured one-qubit QuantumCircuit starting from zero.
# @exception ValueError If repetitions is not a nonnegative integer.
# @note One repetition gives P(0)=P(1)=0.5; two gives P(1)=1.
# @par Example
# sqrt_x_circuit(2) prepares |1>.
def sqrt_x_circuit(repetitions=2):
    if type(repetitions) is not int or repetitions < 0:
        raise ValueError("repetitions must be a nonnegative integer.")
    qc = QuantumCircuit(1)
    for _ in range(repetitions):
        qc.sx(0)
    return qc

## Bell and GHZ State Preparation

Bell states provide entangled reference inputs and correlated random bits.
`bell_state_circuit("phi_plus")` gives 00/11 with equal probability. The minus
variants have the same probabilities but a relative phase detectable by statevector
comparison. GHZ generalizes the same correlation to n qubits.

In [ ]:
## @brief Prepare one of the four Bell states (book example 3-2).
# @param state One of phi_plus, phi_minus, psi_plus, psi_minus.
# @return Unmeasured two-qubit QuantumCircuit.
# @exception ValueError If state is not a supported Bell label.
# @note phi states occupy 00/11; psi states occupy 01/10, each with probability 0.5.
# @par Example
# bell_state_circuit("psi_minus") prepares (|01>-|10>)/sqrt(2), up to global phase.
def bell_state_circuit(state="phi_plus"):
    if state not in ("phi_plus", "phi_minus", "psi_plus", "psi_minus"):
        raise ValueError("state must be phi_plus, phi_minus, psi_plus, or psi_minus.")
    qc = QuantumCircuit(2)
    qc.h(0)
    qc.cx(0, 1)
    if state.startswith("psi"):
        qc.x(0)
    if state.endswith("minus"):
        qc.z(1)
    return qc


## @brief Prepare an n-qubit GHZ state, extending the book's Bell example.
# @param num_qubits Integer at least two.
# @return Unmeasured QuantumCircuit preparing (|0...0>+|1...1>)/sqrt(2).
# @exception ValueError If num_qubits is not an integer at least two.
# @note Only all-zero and all-one strings have nonzero probability, each 0.5.
# @par Example
# ghz_circuit(3) yields 000 or 111.
def ghz_circuit(num_qubits=3):
    if type(num_qubits) is not int or num_qubits < 2:
        raise ValueError("num_qubits must be an integer at least two.")
    qc = QuantumCircuit(num_qubits)
    qc.h(0)
    for target in range(1, num_qubits):
        qc.cx(0, target)
    return qc

## Phase Kickback and Conditional Phase

Phase kickback prepares q0 in |+⟩ and q1 in the phase-gate eigenstate |1⟩.
A controlled phase followed by H on q0 gives `P(q0=0)=cos²(angle/2)`.
The conditional-phase example instead starts in |++⟩: all four probabilities stay
1/4, but |11⟩ gains the phase `exp(i*angle)`. Use amplitude-sensitive tests.

In [ ]:
## @brief Demonstrate an eigenphase transferred to the control (book example 3-3 concept).
# @param angle Finite real phase angle in radians.
# @return Unmeasured two-qubit QuantumCircuit; inspect control q0 and exclude target q1.
# @exception ValueError If angle is not a finite real number.
# @note P(q0=0)=cos(angle/2)**2; unlike the old sample, the target is explicitly |1>.
# @par Example
# phase_kickback_circuit(np.pi) produces displayed state 11.
def phase_kickback_circuit(angle=np.pi):
    _validate_angle(angle, "angle")
    qc = QuantumCircuit(2)
    qc.h(0)
    qc.x(1)
    qc.cp(angle, 0, 1)
    qc.h(0)
    return qc


## @brief Demonstrate a conditional phase on a uniform two-qubit input (examples 3-5 and 5-5).
# @param angle Finite real phase angle in radians applied only to |11>.
# @return Unmeasured two-qubit QuantumCircuit.
# @exception ValueError If angle is not a finite real number.
# @note Statevector is [1,1,1,exp(i*angle)]/2 in Qiskit basis order.
# @par Example
# conditional_phase_circuit(np.pi) reverses the sign of the |11> amplitude.
def conditional_phase_circuit(angle=np.pi):
    _validate_angle(angle, "angle")
    qc = QuantumCircuit(2)
    qc.h([0, 1])
    qc.cp(angle, 0, 1)
    return qc

## Swap Test

Compare single-qubit states `RY(theta_a)|0⟩` and `RY(theta_b)|0⟩`.
The ancilla is q0 and the inputs are q1/q2. In the standard swap-test convention,
`P(q0=0)=(1+|⟨a|b⟩|²)/2`; identical states give 1 and orthogonal states give 1/2.
The book flips its output before measurement; this implementation omits that flip.

In [ ]:
## @brief Estimate state overlap using a controlled SWAP (book example 3-4).
# @param theta_a Finite RY preparation angle in radians for q1.
# @param theta_b Finite RY preparation angle in radians for q2.
# @return Unmeasured three-qubit QuantumCircuit with output ancilla q0.
# @exception ValueError If either preparation angle is not finite and real.
# @note P(q0=0)=(1+cos((theta_a-theta_b)/2)**2)/2. Inspect only q0.
# @par Example
# swap_test_circuit(0, np.pi) yields ancilla probabilities [0.5, 0.5].
def swap_test_circuit(theta_a=0.0, theta_b=0.0):
    _validate_angle(theta_a, "theta_a")
    _validate_angle(theta_b, "theta_b")
    qc = QuantumCircuit(3)
    qc.ry(theta_a, 1)
    qc.ry(theta_b, 2)
    qc.h(0)
    qc.cswap(0, 1, 2)
    qc.h(0)
    return qc

## Coherent Quantum Teleportation

Prepare `|ψ⟩=cos(theta/2)|0⟩+exp(i*phi)sin(theta/2)|1⟩` on q0,
share a Bell pair q1/q2, and transfer the state to q2. Deferred measurement
replaces classical corrections with controlled X/Z. The final state is
`|+⟩[q0]|+⟩[q1]|ψ⟩[q2]`. Inspect Bob's reduced state on q2.
This circuit demonstrates the teleportation identity; it does not implement
classical communication between physically separated parties.

In [ ]:
## @brief Transfer a prepared qubit to q2 with deferred-measurement corrections (chapter 4).
# @param theta Finite Bloch polar preparation angle in radians.
# @param phi Finite Bloch azimuthal preparation angle in radians.
# @return Unmeasured three-qubit QuantumCircuit with Bob's output on q2.
# @exception ValueError If theta or phi is not finite and real.
# @note Verify the reduced state of q2; Z-basis counts alone cannot check the relative phase.
# @par Example
# teleportation_circuit(np.pi, 0) leaves Bob in |1> with probability 1.
def teleportation_circuit(theta=np.pi / 3, phi=np.pi / 4):
    _validate_angle(theta, "theta")
    _validate_angle(phi, "phi")
    qc = QuantumCircuit(3)
    qc.ry(theta, 0)
    qc.p(phi, 0)
    qc.h(1)
    qc.cx(1, 2)
    qc.cx(0, 1)
    qc.h(0)
    qc.cx(1, 2)
    qc.cz(0, 2)
    return qc

## Reversible Integer Arithmetic

All operations wrap modulo `2**num_qubits` and use no scratch qubits.
Set `initial_value=None` on increment to obtain an operation for composition
onto an arbitrary input. Two-register builders prepare basis inputs for reference
tests; composing their operations without that preparation supports superpositions.
Addition and squared addition put destination a on q[0:n] and unchanged source b
on q[n:2n]. Thus the displayed result is `b_bits + a_result_bits`.
Multi-controlled gates can be costly after transpilation; use small examples.

In [ ]:
## @brief Add a signed classical amount modulo 2**n (book example 5-1).
# @param num_qubits Positive destination-register width.
# @param amount Signed integer increment; negative amounts subtract.
# @param initial_value Unsigned basis input, or None to omit preparation.
# @return Unmeasured n-qubit QuantumCircuit; all n qubits form the result register.
# @exception ValueError If width, amount, or initial value is invalid.
# @note Result is (initial_value+amount) modulo 2**n; overflow is intentionally discarded.
# @par Example
# increment_circuit(3, -1, 0) yields 111.
def increment_circuit(num_qubits=3, amount=1, initial_value=0):
    _validate_register_value(num_qubits, 0)
    if type(amount) is not int:
        raise ValueError("amount must be an integer.")
    if initial_value is not None:
        _validate_register_value(num_qubits, initial_value, "initial_value")
    qc = QuantumCircuit(num_qubits)
    targets = list(range(num_qubits))
    if initial_value is not None:
        _prepare_integer(qc, targets, initial_value)
    # Binary expansion of the residue avoids repeated single-step increments.
    residue = amount % 2**num_qubits
    for shift in range(num_qubits):
        if residue & (1 << shift):
            _append_increment(qc, targets, shift=shift)
    return qc


## @brief Reversibly add source b into destination a (book example 5-2).
# @param num_qubits Positive width of each of the two registers.
# @param a Initial unsigned destination value.
# @param b Initial unsigned source value.
# @return Unmeasured 2n-qubit QuantumCircuit; low register is (a+b) modulo 2**n.
# @exception ValueError If width or either basis value is invalid.
# @note Upper register b is unchanged. Reference preparation uses only X gates.
# @par Example
# quantum_addition_circuit(3, 1, 2) yields displayed string 010011.
def quantum_addition_circuit(num_qubits=3, a=1, b=2):
    _validate_register_value(num_qubits, a, "a")
    _validate_register_value(num_qubits, b, "b")
    qc = QuantumCircuit(2 * num_qubits)
    targets = list(range(num_qubits))
    source = list(range(num_qubits, 2 * num_qubits))
    _prepare_integer(qc, targets, a)
    _prepare_integer(qc, source, b)
    for shift, control in enumerate(source):
        _append_increment(qc, targets, [control], shift)
    return qc


## @brief Reversibly add b squared into a modulo 2**n (book example 5-3).
# @param num_qubits Positive width of both registers.
# @param a Initial unsigned destination value.
# @param b Initial unsigned source value.
# @return Unmeasured 2n-qubit QuantumCircuit; q[0:n] holds a+b*b modulo 2**n.
# @exception ValueError If width or either basis value is invalid.
# @note Source q[n:2n] is unchanged. Expansion includes diagonal and doubled cross terms.
# @par Example
# add_squared_circuit(3, 1, 2) yields displayed string 010101.
def add_squared_circuit(num_qubits=3, a=1, b=2):
    _validate_register_value(num_qubits, a, "a")
    _validate_register_value(num_qubits, b, "b")
    qc = QuantumCircuit(2 * num_qubits)
    targets = list(range(num_qubits))
    source = list(range(num_qubits, 2 * num_qubits))
    _prepare_integer(qc, targets, a)
    _prepare_integer(qc, source, b)
    for i, control in enumerate(source):
        _append_increment(qc, targets, [control], 2 * i)
        for j in range(i + 1, num_qubits):
            _append_increment(qc, targets, [control, source[j]], i + j + 1)
    return qc


## @brief Coherently condition modular addition on a flag (book example 5-4 concept).
# @param num_qubits Positive destination-register width, excluding the flag.
# @param amount Signed integer added when the flag is one.
# @param initial_value Unsigned destination input.
# @param control Integer 0 or 1, or None to prepare a coherent |+> flag.
# @return Unmeasured (n+1)-qubit QuantumCircuit; q0 is flag, q[1:n+1] is destination.
# @exception ValueError If any width, amount, input or flag setting is invalid.
# @note With control=None, the output coherently correlates the flag with the two branches.
# @par Example
# conditional_increment_circuit(2, 1, 0, None) yields 000/011, each with probability 0.5.
def conditional_increment_circuit(num_qubits=3, amount=1, initial_value=0, control=1):
    _validate_register_value(num_qubits, initial_value, "initial_value")
    if type(amount) is not int:
        raise ValueError("amount must be an integer.")
    if control is not None and (type(control) is not int or control not in (0, 1)):
        raise ValueError("control must be 0, 1, or None.")
    qc = QuantumCircuit(num_qubits + 1)
    if control is None:
        qc.h(0)
    elif control == 1:
        qc.x(0)
    targets = list(range(1, num_qubits + 1))
    _prepare_integer(qc, targets, initial_value)
    residue = amount % 2**num_qubits
    for shift in range(num_qubits):
        if residue & (1 << shift):
            _append_increment(qc, targets, [0], shift)
    return qc


## @brief Compute reversible XOR while retaining its inputs (book example 5-6 concept).
# @param a First input bit as integer 0 or 1.
# @param b Second input bit as integer 0 or 1.
# @return Unmeasured three-qubit QuantumCircuit; q0=a, q1=b, q2=a XOR b.
# @exception ValueError If either input is not an integer bit.
# @note The zero-initialized output is separate from the preserved inputs.
# @par Example
# reversible_xor_circuit(1, 0) yields displayed string 101.
def reversible_xor_circuit(a=1, b=0):
    _validate_register_value(1, a, "a")
    _validate_register_value(1, b, "b")
    qc = QuantumCircuit(3)
    _prepare_integer(qc, [0, 1], a + 2 * b)
    qc.cx(0, 2)
    qc.cx(1, 2)
    return qc

## Grover Search with Multiple Solutions

Mark a collection of equally sized distinct bitstrings. If M of N states are
marked, the total marked probability after r iterations is
`sin²((2r+1)*asin(sqrt(M/N)))`. All marked states have equal probability.
The default three-qubit example marks 000 and 111; one iteration gives total
marked probability 1. Explicit iteration counts also demonstrate overrotation.

In [ ]:
## @brief Amplify multiple marked basis states (book example 6-3).
# @param marked_states Nonempty list or tuple of distinct, equal-length binary strings.
# @param iterations Nonnegative integer, or None for floor(pi/(4*asin(sqrt(M/N)))).
# @return Unmeasured n-qubit QuantumCircuit with no auxiliary register.
# @exception ValueError If marked strings or iteration count are invalid.
# @note Read all qubits. All-marked input uses zero iterations by default.
# @par Example
# grover_multi_circuit(("000", "111")) yields 000/111, each with probability 0.5.
def grover_multi_circuit(marked_states=("000", "111"), iterations=None):
    if not isinstance(marked_states, (list, tuple)) or not marked_states:
        raise ValueError("marked_states must be a nonempty list or tuple.")
    if any(not isinstance(s, str) or not s or any(bit not in "01" for bit in s) for s in marked_states):
        raise ValueError("marked states must be nonempty binary strings.")
    size = len(marked_states[0])
    if any(len(s) != size for s in marked_states) or len(set(marked_states)) != len(marked_states):
        raise ValueError("marked states must be distinct and have equal lengths.")
    if iterations is None:
        theta = np.arcsin(np.sqrt(len(marked_states) / 2**size))
        iterations = int(np.floor(np.pi / (4 * theta)))
    if type(iterations) is not int or iterations < 0:
        raise ValueError("iterations must be a nonnegative integer or None.")
    qc = QuantumCircuit(size)
    qubits = list(range(size))
    qc.h(qubits)
    for _ in range(iterations):
        for state in marked_states:
            zeros = [q for q, bit in enumerate(reversed(state)) if bit == "0"]
            for q in zeros:
                qc.x(q)
            _append_phase_flip(qc, qubits)
            for q in zeros:
                qc.x(q)
        qc.h(qubits)
        qc.x(qubits)
        _append_phase_flip(qc, qubits)
        qc.x(qubits)
        qc.h(qubits)
    return qc

## QFT Signal Analysis

Prepare a phase ramp `exp(2πi*frequency*x/N)/sqrt(N)` then apply inverse QFT
to recover its integer frequency. The square mode uses a normalized square pulse
with amplitudes on the first half of the basis states, then applies forward QFT.
The book calls the negative-phase transform QFT; this library explicitly calls it
inverse QFT to stay consistent with the existing positive-phase `qft_circuit`.
These are representative chapter 7 signals, not replicas of each plotted waveform.

In [ ]:
## @brief Analyze a phase-ramp or square-pulse input with QFT (chapter 7 concepts).
# @param num_qubits Positive signal-register width.
# @param frequency Unsigned integer frequency below 2**n; used only in phase mode.
# @param signal Either phase or square; square requires frequency=0 to avoid ignored inputs.
# @return Unmeasured n-qubit QuantumCircuit including signal preparation and transform.
# @exception ValueError If width, frequency or signal selection is invalid.
# @note Phase mode yields frequency deterministically; square mode has a Fourier distribution.
# @par Example
# qft_signal_circuit(3, 3) yields 011 with probability 1.
def qft_signal_circuit(num_qubits=3, frequency=1, signal="phase"):
    _validate_register_value(num_qubits, frequency, "frequency")
    if signal not in ("phase", "square"):
        raise ValueError("signal must be phase or square.")
    if signal == "square" and frequency != 0:
        raise ValueError("frequency must be zero for square mode.")
    qc = QuantumCircuit(num_qubits)
    if signal == "phase":
        qc.h(range(num_qubits))
        for qubit in range(num_qubits):
            qc.p(2 * np.pi * frequency * 2**qubit / 2**num_qubits, qubit)
        qc.compose(qft_circuit(num_qubits, inverse=True), inplace=True)
    else:
        # q[n-1]=0, lower qubits uniform: first half of basis amplitudes are equal.
        for qubit in range(num_qubits - 1):
            qc.h(qubit)
        qc.compose(qft_circuit(num_qubits), inplace=True)
    return qc

## Reference Outputs for the Added Builders

| Call | Register to inspect | Ideal result |
|---|---|---|
| `sqrt_x_circuit()` | q0 | 1 with probability 1 |
| `bell_state_circuit()` | q0, q1 | 00/11, each 0.5 |
| `ghz_circuit()` | q0, q1, q2 | 000/111, each 0.5 |
| `phase_kickback_circuit()` | q0 | 1 with probability 1 |
| `conditional_phase_circuit()` | full statevector | [1, 1, 1, -1]/2 |
| `swap_test_circuit()` | q0 | 0 with probability 1 |
| `teleportation_circuit(theta, phi)` | reduced state of q2 | prepared input state |
| `increment_circuit()` | all qubits | 001 with probability 1 |
| `quantum_addition_circuit()` | q0–q2 | 011 with probability 1 |
| `add_squared_circuit()` | q0–q2 | 101 with probability 1 |
| `conditional_increment_circuit()` | q1–q3 | 001 with probability 1 |
| `reversible_xor_circuit()` | q2 | 1 with probability 1 |
| `grover_multi_circuit()` | all qubits | 000/111, each 0.5 |
| `qft_signal_circuit()` | all qubits | 001 with probability 1 |

Example after loading the definition cells:

```python
qc = quantum_addition_circuit(3, a=1, b=2)
probabilities = Statevector.from_instruction(qc).probabilities([0, 1, 2])
assert np.isclose(probabilities[3], 1.0)
```

Computational-basis counts do not check relative phase. The companion validation
script checks amplitudes, reduced-state fidelity, arithmetic on coherent inputs,
and analytic distributions. Run from the repository root:

```sh
.venv/bin/python -m unittest discover -s tests -p 'test_quantum_library.py' -v
python3 local_doxygen/refresh_sources.py
doxygen local_doxygen/Doxyfile
```